# Tool-RL SFT (Phi-2) - Google Colab



This notebook recreates the minimal project files needed to run SFT in Colab using microsoft/phi-2 as the starting model.

In [ ]:
!pip -q install torch transformers datasets accelerate peft sentencepiece safetensors numpy pandas tqdm pyyaml

In [ ]:
from pathlib import Path
import os
import torch

Path('src').mkdir(parents=True, exist_ok=True)
Path('experiments').mkdir(parents=True, exist_ok=True)
Path('data/raw/apigen_mt5k').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile src/data.py
import argparse
import json
import random
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

from datasets import Dataset, load_dataset

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "apigen_mt5k"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"


def load_apigen(split: str = "train") -> Dataset:
    """Load the requested APIGen-MT-5k split from Hugging Face."""
    return load_dataset("Salesforce/APIGen-MT-5k", split=split)


def dump_raw_split(dataset: Dataset, split: str) -> Path:
    """Persist the raw split as JSONL inside data/raw for reproducibility."""
    RAW_PATH.mkdir(parents=True, exist_ok=True)
    out_path = RAW_PATH / f"{split}.jsonl"
    with out_path.open("w", encoding="utf-8") as fp:
        for row in dataset:
            fp.write(json.dumps(row, ensure_ascii=True) + "\n")
    return out_path


def extract_tool_call(messages: Sequence[Dict]) -> Dict | None:
    """Return the first tool call (function_call from="" message value) if present."""
    for message in messages:
        if message.get("from", "").lower() == "function_call":
            try:
                call_dict = json.loads(message.get("value", "{}"))
                if call_dict.get("name"):
                    return call_dict
            except (json.JSONDecodeError, TypeError):
                continue
    return None


def extract_latest_user_message(messages: Sequence[Dict]) -> str | None:
    """Return the most recent user (human) message content, if available."""
    user_messages = [m for m in messages if m.get("from", "").lower() == "human"]
    if not user_messages:
        return None
    return user_messages[-1].get("value", "").strip()


def format_prompt(tools: Sequence[Dict], user_message: str) -> str:
    """Construct the textual prompt shown to the model."""
    tools_json = json.dumps(tools, indent=2)
    return (
        "SYSTEM: You have access to these tools:\n"
        f"{tools_json}\n\n"
        f"USER: {user_message}\n"
        "ASSISTANT:"
    )


def format_example(example: Dict) -> Dict | None:
    """Convert a raw dataset record into a prompt/target pair."""
    tools = example.get("tools", [])
    conversations = example.get("conversations", [])
    tool_call = extract_tool_call(conversations)
    user_message = extract_latest_user_message(conversations)
    if not tool_call or not user_message:
        return None

    prompt = format_prompt(tools, user_message)
    target = json.dumps(
        {
            "name": tool_call.get("name"),
            "arguments": tool_call.get("arguments"),
        },
        ensure_ascii=True,
    )
    return {"prompt": prompt, "target": target}


def preprocess_dataset(dataset: Iterable[Dict]) -> List[Dict]:
    """Apply formatting and drop records that lack tool calls."""
    processed: List[Dict] = []
    for record in dataset:
        formatted = format_example(record)
        if formatted:
            processed.append(formatted)
    return processed


def split_train_val(
    examples: List[Dict],
    val_ratio: float,
    seed: int,
) -> Tuple[List[Dict], List[Dict]]:
    """Shuffle and split processed examples into train/val subsets."""
    if not examples or val_ratio <= 0:
        return examples, []

    rng = random.Random(seed)
    shuffled = examples.copy()
    rng.shuffle(shuffled)
    val_count = max(1, int(len(shuffled) * val_ratio))
    val_examples = shuffled[:val_count]
    train_examples = shuffled[val_count:]
    return train_examples, val_examples


def write_jsonl(records: Sequence[Dict], path: Path) -> None:
    """Write list of dicts as JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fp:
        for record in records:
            fp.write(json.dumps(record, ensure_ascii=True) + "\n")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Download + preprocess APIGen-MT-5k")
    parser.add_argument(
        "--splits",
        nargs="+",
        default=["train"],
        help="HF dataset splits to download (default: train)",
    )
    parser.add_argument(
        "--val-ratio",
        type=float,
        default=0.05,
        help="Validation ratio when splitting the train split",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Seed used for the train/val split",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    for split in args.splits:
        dataset = load_apigen(split)
        raw_path = dump_raw_split(dataset, split)
        processed_records = preprocess_dataset(dataset)

        if split == "train":
            train_records, val_records = split_train_val(
                processed_records, args.val_ratio, args.seed
            )
            write_jsonl(train_records, PROCESSED_PATH / "train.jsonl")
            if val_records:
                write_jsonl(val_records, PROCESSED_PATH / "val.jsonl")
        else:
            write_jsonl(processed_records, PROCESSED_PATH / f"{split}.jsonl")

        print(
            f"Saved raw {split} split to {raw_path.relative_to(PROJECT_ROOT)} "
            f"({len(dataset)} rows)"
        )


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train_sft.py
"""Supervised fine-tuning entrypoint."""
from __future__ import annotations

import argparse
import inspect
from pathlib import Path
from typing import Any, Dict, List

import torch
import yaml
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = Path(__file__).resolve().parents[1]


def resolve_torch_dtype(use_fp16: bool, use_bf16: bool) -> torch.dtype | None:
    if use_bf16:
        return torch.bfloat16
    if use_fp16:
        return torch.float16
    return None


def load_config(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as fp:
        return yaml.safe_load(fp)


def get_cfg(cfg: Dict[str, Any], *keys: str, default: Any) -> Any:
    cursor: Any = cfg
    for key in keys:
        if not isinstance(cursor, dict) or key not in cursor:
            return default
        cursor = cursor[key]
    return cursor


def resolve_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def build_tokenizer(name: str) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True)
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
    return tokenizer


def tokenize_examples(
    examples: Dict[str, List[str]],
    tokenizer: AutoTokenizer,
    max_length: int,
) -> Dict[str, List[List[int]]]:
    input_ids_list: List[List[int]] = []
    attention_masks: List[List[int]] = []
    labels_list: List[List[int]] = []

    eos_id = tokenizer.eos_token_id
    for prompt, target in zip(examples["prompt"], examples["target"]):
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
            verbose=False,
        ).input_ids
        target_ids = tokenizer(
            target,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
            verbose=False,
        ).input_ids
        if eos_id is not None:
            target_ids = target_ids + [eos_id]

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        if max_length and len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
            labels = labels[:max_length]

        if len(labels) < len(input_ids):
            labels = labels + [-100] * (len(input_ids) - len(labels))
        elif len(labels) > len(input_ids):
            labels = labels[:len(input_ids)]

        attention_mask = [1] * len(input_ids)
        input_ids_list.append(input_ids)
        attention_masks.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list,
    }


def collate_batch(batch: List[Dict[str, Any]], tokenizer: AutoTokenizer) -> Dict[str, Any]:
    """Collate batch by padding to max length in batch."""
    if not batch:
        raise ValueError("Cannot collate empty batch")

    max_len = max(len(item["input_ids"]) for item in batch)
    if max_len == 0:
        raise ValueError("Empty sequences in batch")

    pad_id = tokenizer.pad_token_id
    input_ids: List[List[int]] = []
    attention_masks: List[List[int]] = []
    labels: List[List[int]] = []

    for item in batch:
        length = len(item["input_ids"])
        pad_size = max_len - length
        input_ids.append(item["input_ids"] + [pad_id] * pad_size)
        attention_masks.append(item["attention_mask"] + [0] * pad_size)
        labels.append(item["labels"] + [-100] * pad_size)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train SFT model for tool use")
    parser.add_argument("--config", type=Path, required=True, help="Path to YAML config")
    parser.add_argument("--model-name", default=None, help="Override model name")
    parser.add_argument("--tokenizer-name", default=None, help="Override tokenizer name")
    parser.add_argument("--max-train-samples", type=int, default=None)
    parser.add_argument("--max-eval-samples", type=int, default=None)
    parser.add_argument("--max-steps", type=int, default=None)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    config = load_config(args.config)

    seed = get_cfg(config, "seed", default=42)
    set_seed(seed)

    model_name = args.model_name or get_cfg(config, "model", "model_id", default=None)
    if model_name is None:
        model_name = get_cfg(config, "model_name", default="gpt2")

    tokenizer_name = args.tokenizer_name or get_cfg(config, "model", "tokenizer_id", default=None)
    if tokenizer_name is None:
        tokenizer_name = get_cfg(config, "tokenizer_name", default=model_name)

    output_root = resolve_path(get_cfg(config, "logging", "output_dir", default="experiments/runs"))
    run_name = get_cfg(config, "run_name", default="sft")
    output_dir = output_root / run_name

    train_path = resolve_path(get_cfg(config, "data", "processed_path", default="data/processed/train.jsonl"))
    eval_path = resolve_path(get_cfg(config, "data", "eval_path", default="data/processed/val.jsonl"))
    if not train_path.exists():
        raise FileNotFoundError(f"Training data not found: {train_path}")

    data_files: Dict[str, str] = {"train": str(train_path)}
    if eval_path.exists():
        data_files["validation"] = str(eval_path)

    dataset: DatasetDict = load_dataset("json", data_files=data_files)
    if args.max_train_samples:
        dataset["train"] = dataset["train"].select(range(args.max_train_samples))
    if args.max_eval_samples and "validation" in dataset:
        dataset["validation"] = dataset["validation"].select(range(args.max_eval_samples))

    tokenizer = build_tokenizer(tokenizer_name)
    max_length = get_cfg(config, "train", "max_length", default=1024)

    tokenized = dataset.map(
        lambda batch: tokenize_examples(batch, tokenizer, max_length),
        batched=True,
        remove_columns=dataset["train"].column_names,
    )

    per_device_batch_size = get_cfg(config, "train", "per_device_batch_size", default=2)
    gradient_accumulation_steps = get_cfg(config, "train", "gradient_accumulation_steps", default=1)
    learning_rate = get_cfg(config, "train", "learning_rate", default=5e-5)
    num_epochs = get_cfg(config, "train", "sft_epochs", default=1)
    log_interval = get_cfg(config, "logging", "log_interval", default=50)
    warmup_steps = get_cfg(config, "train", "warmup_steps", default=100)
    use_fp16 = get_cfg(config, "train", "fp16", default=False)
    use_bf16 = get_cfg(config, "train", "bf16", default=False)
    gradient_checkpointing = get_cfg(config, "train", "gradient_checkpointing", default=False)
    trust_remote_code = get_cfg(config, "model", "trust_remote_code", default=False)
    torch_dtype = resolve_torch_dtype(use_fp16, use_bf16)

    model_config = AutoConfig.from_pretrained(
        model_name,
        trust_remote_code=trust_remote_code,
    )
    if not hasattr(model_config, "pad_token_id") or model_config.pad_token_id is None:
        model_config.pad_token_id = tokenizer.pad_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=trust_remote_code,
        dtype=torch_dtype,
        config=model_config,
    )
    model_param_dtype = next(model.parameters()).dtype
    trainer_fp16 = use_fp16 and model_param_dtype != torch.float16
    if use_fp16 and not trainer_fp16:
        print(
            "Model weights are already float16; disabling Trainer fp16 ",
            "to avoid GradScaler unscale errors."
        )
    if not hasattr(model.config, "pad_token_id") or model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False

    if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
        model.resize_token_embeddings(len(tokenizer))

    training_kwargs: Dict[str, Any] = {
        "output_dir": str(output_dir),
        "num_train_epochs": num_epochs,
        "per_device_train_batch_size": per_device_batch_size,
        "per_device_eval_batch_size": per_device_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": learning_rate,
        "warmup_steps": warmup_steps,
        "logging_steps": log_interval,
        "eval_steps": log_interval if "validation" in tokenized else None,
        "save_strategy": "steps",
        "save_steps": log_interval * 5,
        "save_total_limit": 3,
        "max_steps": args.max_steps if args.max_steps else -1,
        "fp16": trainer_fp16,
        "bf16": use_bf16,
        "gradient_checkpointing": gradient_checkpointing,
        "report_to": [],
        "load_best_model_at_end": True if "validation" in tokenized else False,
        "metric_for_best_model": "eval_loss" if "validation" in tokenized else None,
    }

    strategy_value = "steps" if "validation" in tokenized else "no"
    training_params = inspect.signature(TrainingArguments.__init__).parameters
    if "evaluation_strategy" in training_params:
        training_kwargs["evaluation_strategy"] = strategy_value
    else:
        training_kwargs["eval_strategy"] = strategy_value

    training_args = TrainingArguments(**training_kwargs)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized.get("validation"),
        data_collator=lambda batch: collate_batch(batch, tokenizer),
    )

    print(f"\n{'='*60}")
    print(f"Starting training with:")
    print(f"  Model: {model_name}")
    print(f"  Train samples: {len(tokenized['train'])}")
    print(f"  Eval samples: {len(tokenized.get('validation', []))}")
    print(f"  Batch size: {per_device_batch_size}")
    print(f"  Grad accumulation: {gradient_accumulation_steps}")
    print(f"  Effective batch: {per_device_batch_size * gradient_accumulation_steps}")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Max steps: {args.max_steps if args.max_steps else 'None'}")
    print(f"  Gradient checkpointing: {gradient_checkpointing}")
    print(f"  Trust remote code: {trust_remote_code}")
    print(f"  Output dir: {output_dir}")
    print(f"{'='*60}\n")

    trainer.train()

    final_dir = output_dir / "final"
    print(f"\nSaving model to {final_dir}")
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f"Training complete!")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments/config.colab.phi2.yaml
run_name: colab_phi2
seed: 42

model:
  model_id: "microsoft/phi-2"
  trust_remote_code: true
  tokenizer_id: "microsoft/phi-2"

model_name: "microsoft/phi-2"
tokenizer_name: "microsoft/phi-2"

train:
  sft_epochs: 1
  per_device_batch_size: 1
  gradient_accumulation_steps: 8
  gradient_checkpointing: true
  learning_rate: 5.0e-5
  warmup_steps: 100
  max_length: 1024
  fp16: true
  bf16: false

data:
  processed_path: "data/processed/train.jsonl"
  eval_path: "data/processed/val.jsonl"

logging:
  output_dir: "experiments/runs"
  log_interval: 20

In [ ]:
!python src/data.py --splits train --val-ratio 0.05 --seed 42

In [ ]:
!python src/train_sft.py --config experiments/config.colab.phi2.yaml --max-train-samples 1000 --max-eval-samples 100 --max-steps 200

In [ ]:
!ls -lah experiments/runs/colab_phi2/final